In [1]:
"""
stage4_mc_dropout_uncertainty.py
============================================================================
Kaggle Notebook. GPU strongly recommended (training a DenseNet-121);
map generation can run on CPU if needed but will be slow.

Stage 4 of the thesis plan: trains a DenseNet-121 multi-label chest X-ray
classifier with internal dropout, then uses Monte Carlo Dropout (N
stochastic forward passes at inference, dropout active / BatchNorm frozen)
combined with Grad-CAM to produce a per-image SPATIAL uncertainty map --
directly comparable to the diffusion-ensemble variance maps from Stage 1,
enabling a head-to-head comparison of the two uncertainty methods.

DESIGN DECISION (the guide leaves this implicit -- documented here so it's
a stated methods decision, not a silent assumption): MC-Dropout on a
classifier alone gives one scalar uncertainty per image, not a spatial
map. This script extracts a map via Grad-CAM, computed once per stochastic
forward pass, then measures pixel-wise VARIANCE across the N Grad-CAM maps
as the MC-Dropout "uncertainty map" -- the classifier-based analogue of
Stage 1's diffusion-ensemble pixel variance.

OUTPUT FORMAT MATCHES STAGE 1 DELIBERATELY: uncertainty_chunk_NNN.npz with
keys "{image_id}__mean" / "{image_id}__variance" (both (512,512) float32),
plus a companion _meta.csv -- identical to
ddim_epistemic_uncertainty_pipeline.py's output. This means
calibration_spatial_evaluation_pipeline.py (Stage 3) can score these maps
with ZERO code changes: just point NPZ_DIR at this script's OUTPUT_DIR.

TWO SEPARATE ENTRY POINTS, meant to run in separate Kaggle cells /
sessions (training is checkpointed for the 9-hour Kaggle session limit,
per the thesis plan's own checkpointing requirement):
  1. train_classifier_entrypoint()      -- trains and checkpoints the model
  2. generate_uncertainty_maps_entrypoint() -- loads the trained checkpoint,
     generates MC-Dropout+Grad-CAM variance maps for a given class's
     image_ids (read directly from that class's Stage 1 npz output, so
     the exact same images are used -- see get_stage1_image_ids())

A CRITICAL, EASY-TO-GET-WRONG DETAIL this script handles explicitly:
naive MC-Dropout implementations call `.train()` on dropout submodules,
which RECURSIVELY re-enables their child BatchNorm layers too --
corrupting BatchNorm statistics during inference. enable_mc_dropout()
below sets `.training = True` as a direct attribute (bypassing the
recursive `.train()` method) so ONLY dropout is stochastic while
BatchNorm stays frozen in eval mode. This was verified empirically before
writing this script, not assumed.
"""

# ============================================================================
# CELL 0 — Install dependencies (Kaggle: internet must be ON for
# pretrained DenseNet-121 weights to download from download.pytorch.org)
# ============================================================================
try:
    import h5py
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "h5py"], check=True)
    raise SystemExit("h5py installed. RESTART THE KERNEL, then re-run this cell.")

import time
import warnings
from ast import literal_eval
from pathlib import Path

import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.models as models
from tqdm.auto import tqdm


# ============================================================================
# CELL 1 — CONFIG
# ============================================================================
H5_PATH = "/kaggle/input/datasets/pgc17ms072/vindr-cxr-512-h5/vindr_cxr_512.h5"
METADATA_CSV_PATH = "/kaggle/input/datasets/pgc17ms072/vindr-cxr-512-h5/vindr_cxr_metadata.csv"

# Same 5 classes used across Stages 1-3.
TARGET_CLASSES = ["Pneumothorax", "Consolidation", "Nodule/Mass", "Cardiomegaly", "Atelectasis"]

# Stage 1's npz output directory for whichever class you're currently
# generating MC-Dropout maps for (see get_stage1_image_ids()) -- this
# guarantees Stage 4 evaluates the IDENTICAL image_ids Stage 1 used for
# that class, required for a fair head-to-head comparison.
# Where each class's cohort (the exact image_ids Stage 1 processed) comes
# from. Stage 4 NEVER reads the diffusion variance arrays -- it only needs
# the image_id LIST, so MC-Dropout runs on the identical cohort. Three
# accepted forms, per class:
#
#   1. A directory of uncertainty_chunk_*.npz  (the usual case)
#   2. A LIST of such directories -- for a class whose generation spanned
#      several Kaggle sessions, each writing its own output folder
#   3. A path to a .csv with an `image_id` column -- simplest when you have
#      already combined multi-session metadata into one file. This is the
#      recommended route for Cardiomegaly, whose combined metadata CSV
#      already defines the cohort, avoiding the need to touch 33 npz files
#      spread over 3 directories.
STAGE1_NPZ_DIR_BY_CLASS = {
    "Pneumothorax": "/kaggle/input/datasets/pgc17ms072/pneumothorax-uncertainity-train-control",
    "Consolidation": "/kaggle/input/datasets/pgc17ms072/consolidation-gen-train",
    "Atelectasis": "/kaggle/input/datasets/kartikichandratre/atelectasis-uncertainity-train-eta1-n20-g1",
    "Nodule/Mass": "/kaggle/input/datasets/pgc17ms072/nodule-uncertainity-train-eta1-n20-g1",
    # Multi-session class -> point at the combined metadata CSV:
    "Cardiomegaly": "/kaggle/input/datasets/kartikichandratre/cardiomegaly-combined-train-metadata/cardiomegaly_combined_meta.csv",
    # ...or at every session directory:
    # "Cardiomegaly": [
    #     "/kaggle/input/cardiomegaly-uncertainity-train-eta1-n20-g1-sess0",
    #     "/kaggle/input/cardiomegaly-unceratainty-train-eta1-n20-g1-sess1",
    #     "/kaggle/input/cardiomelgaly-uncertainity-sess2",
    # ],
}

CHECKPOINT_DIR = ""

# Where map generation LOADS the trained model from. Leave as None to use
# CHECKPOINT_DIR (i.e. a model trained in this same session).
#
# Set this when the model was trained in an EARLIER session and saved as a
# Kaggle Dataset -- /kaggle/working does not persist, so a new session has
# to read the weights from /kaggle/input instead. Keep it SEPARATE from
# CHECKPOINT_DIR: /kaggle/input is read-only, so pointing CHECKPOINT_DIR
# there would break training's ability to write checkpoints.
#
# Accepts either the .pt file directly, or the directory containing it:
#   "/kaggle/input/densenet-mc-dropout/densenet121_mc_dropout_best.pt"
#   "/kaggle/input/densenet-mc-dropout"
INFERENCE_CHECKPOINT_PATH = "/kaggle/input/datasets/kartikichandratre/stage4-densenet121-mc-dropout-best-model"
OUTPUT_ROOT = "/kaggle/working/stage4_mc_dropout_maps"  # one subfolder per class

IMG_SIZE = 512             # native resolution of your h5-stored images / output maps

# --- Uncertainty method -----------------------------------------------------
# "gradcam" : variance across N stochastic Grad-CAM attribution maps.
#             Measures disagreement about WHICH REGION drove the prediction.
# "logit"   : variance across N stochastic per-pixel logit maps, obtained by
#             applying the classifier convolutionally instead of after global
#             average pooling. Measures disagreement about the PREDICTION
#             ITSELF at each location -- conceptually the closer analogue to
#             the diffusion ensemble's pixel-intensity variance, and cheaper
#             (no backward pass needed).
# Running BOTH and comparing is the strongest thesis result: it separates
# "is MC-Dropout worse?" from "is attribution-variance the wrong quantity?".
UNCERTAINTY_METHOD = "logit"     # "gradcam" | "logit"

# --- Resolution -------------------------------------------------------------
# Measured native feature-map sizes (DenseNet-121):
#            input 224     input 512
#   norm5      7x7           16x16
#   denseblock3 14x14        32x32
#   denseblock2 28x28        64x64
# Default below (denseblock3 @ 512 input) gives 32x32 -- a 4.6x linear
# resolution gain over the original norm5 @ 224 (7x7) setup, for ~3.5x
# compute. Diffusion maps are natively 512x512, so closing this gap matters:
# Dice in particular punishes boundary imprecision hard.
MODEL_INPUT_SIZE = 512
GRADCAM_TARGET_LAYER = "features.denseblock3"

# --- CAM normalization toggle ----------------------------------------------
# True : min-max normalize EACH pass's CAM to [0,1] before accumulating
#        variance. Variance then reflects SHAPE disagreement only -- if all
#        passes agree on location but disagree on intensity, variance ~ 0.
# False: accumulate raw (un-normalized) CAMs, so variance reflects BOTH
#        shape and magnitude disagreement.
# This is a real methodological choice, not a formatting detail -- state
# whichever you use in your Methods. Ignored when UNCERTAINTY_METHOD="logit"
# (logit maps are never per-pass normalized; that would destroy the signal).
NORMALIZE_CAM_PER_PASS = False

DROP_RATE = 0.3            # dropout probability, both internal DenseNet blocks + classifier head
# --- Training memory / throughput at 512px ---------------------------------
# At MODEL_INPUT_SIZE=512 a DenseNet-121 forward+backward costs ~0.72 GB per
# sample (measured), so BATCH_SIZE=16 would need ~11.5 GB before optimizer
# state and CUDA context -- too close to a 16 GB T4's limit.
#
# BATCH_SIZE x ACCUMULATION_STEPS is the EFFECTIVE batch: gradients are
# accumulated over that many micro-batches before each optimizer.step(), so
# optimization dynamics match a single large batch while peak memory only
# ever holds BATCH_SIZE samples. Defaults below give an effective batch of
# 16 -- if you still OOM, halve BATCH_SIZE and double ACCUMULATION_STEPS
# (the effective batch, and therefore the training behaviour, is unchanged).
BATCH_SIZE = 8
ACCUMULATION_STEPS = 2     # effective batch = BATCH_SIZE * ACCUMULATION_STEPS

# Automatic Mixed Precision: the T4 has fp16 tensor cores, so AMP roughly
# halves activation memory and typically gives 1.5-2x speedup at 512px.
# Consistent with the diffusion pipeline, which already runs torch.float16.
# Automatically inert on CPU (where fp16 autocast would be slower).
USE_AMP = True

# At 512px each image is ~1 MB uncompressed and the h5 is gzip-compressed,
# so DataLoader decompression can become the bottleneck rather than the GPU.
# If you see low GPU utilization during training, raise this further.
NUM_WORKERS = 4

# MC passes for one image run in BATCHES of this many stochastic replicas
# rather than one at a time. Verified correct: with BatchNorm in eval mode
# there is no cross-sample interaction, and dropout draws independent masks
# per batch element, so summing the per-sample logits and calling backward
# ONCE yields gradients bitwise identical to sequential passes (tested:
# max abs diff exactly 0.0).
#
# MEMORY (measured, DenseNet-121 @ 512px input, with backward):
#   batch 2 -> 2.2 GB      batch 4 -> 3.6 GB      ~0.72 GB per extra sample
# So batching all 20 at once at 512px would need ~15 GB and WILL OOM a
# 16 GB T4. Default 4 is the safe choice at 512px: still 5x fewer kernel
# launches than sequential, with headroom. Raise toward 20 if you drop
# MODEL_INPUT_SIZE to 224 (~5x less activation memory), or when using
# UNCERTAINTY_METHOD="logit" (no backward -> much lower peak).
MC_BATCH_SIZE = 4
NUM_EPOCHS = 15
LEARNING_RATE = 1e-4
VAL_FRACTION = 0.15
N_MC_PASSES = 20           # matches the diffusion ensemble's baseline N=20 for a fair comparison
CHUNK_SIZE = 50            # images per output npz chunk, matches Stage 1's convention
RANDOM_SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ============================================================================
# CELL 2 — Dataset
# ============================================================================
class VinDrCXRMultiLabelDataset(Dataset):
    """
    Reads (512,512) float32 images in [-1,1] from the h5 file produced by
    vinbigdata_diffusion_pipeline.py, resizes to MODEL_INPUT_SIZE and
    replicates to 3 channels for DenseNet-121, and builds a multi-label
    target vector over TARGET_CLASSES from the metadata's `labels` column.
    """
    def __init__(self, h5_path, metadata_df, target_classes,
                 model_input_size=MODEL_INPUT_SIZE):
        self.h5_path = h5_path
        self.metadata_df = metadata_df.reset_index(drop=True)
        self.target_classes = target_classes
        self.model_input_size = model_input_size
        self._h5file = None  # opened lazily per worker process

    def _h5(self):
        if self._h5file is None:
            self._h5file = h5py.File(self.h5_path, "r")
        return self._h5file

    def __len__(self):
        return len(self.metadata_df)

    def _load_and_resize(self, image_id):
        arr = self._h5()[image_id][()].astype(np.float32)  # (512, 512), [-1, 1]
        arr_t = torch.from_numpy(arr)[None, None, :, :]
        arr_t = F.interpolate(arr_t, size=(self.model_input_size, self.model_input_size),
                               mode="bilinear", align_corners=False)
        return arr_t.squeeze(0).repeat(3, 1, 1)  # (3, H, W)

    def __getitem__(self, idx):
        row = self.metadata_df.iloc[idx]
        image_id = row["image_id"]
        image_tensor = self._load_and_resize(image_id)

        labels = row["labels"]
        if isinstance(labels, str):
            labels = literal_eval(labels)
        target = torch.zeros(len(self.target_classes), dtype=torch.float32)
        for i, cls in enumerate(self.target_classes):
            if cls in labels:
                target[i] = 1.0

        return image_tensor, target, image_id


# ============================================================================
# CELL 3 — Model: DenseNet-121 with dropout, + correct MC-Dropout toggle
# ============================================================================
def build_densenet121_mc_dropout(num_classes, drop_rate=DROP_RATE, pretrained=True):
    """
    drop_rate > 0 activates torchvision DenseNet's INTERNAL dropout inside
    each dense layer (applied functionally, gated by that submodule's own
    `.training` flag -- there is no separate nn.Dropout submodule for it,
    verified against torchvision source before writing this). We also add
    an explicit nn.Dropout in the classifier head.
    """
    weights = models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.densenet121(weights=weights, drop_rate=drop_rate)
    in_features = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=drop_rate),
        nn.Linear(in_features, num_classes),
    )
    return model


def enable_mc_dropout(model):
    """
    Sets the WHOLE model to eval() first (freezes BatchNorm running
    stats), then re-enables stochastic dropout ONLY -- as a direct
    `.training = True` ATTRIBUTE assignment, not via `.train()` (which
    would recursively re-enable each dropout-capable submodule's own
    BatchNorm children too -- verified empirically to be a real bug
    before writing this).

    Targets both DenseNet's internal dense-layer dropout (any submodule
    with a `drop_rate` attribute > 0) and any explicit nn.Dropout module
    in the classifier head.
    """
    model.eval()
    n_enabled = 0
    for module in model.modules():
        has_internal_dropout = getattr(module, "drop_rate", 0) and module.drop_rate > 0
        is_explicit_dropout = isinstance(module, (nn.Dropout, nn.Dropout2d, nn.Dropout3d))
        if has_internal_dropout or is_explicit_dropout:
            module.training = True
            n_enabled += 1
    if n_enabled == 0:
        raise RuntimeError(
            "enable_mc_dropout found no dropout-capable modules -- check "
            "the model was built with DROP_RATE > 0 via "
            "build_densenet121_mc_dropout()."
        )
    return n_enabled


# ============================================================================
# CELL 4 — Training (checkpointed for Kaggle's 9-hour session limit)
# ============================================================================
def build_train_val_loaders(h5_path, metadata_csv_path, target_classes,
                             batch_size, val_fraction, seed):
    metadata = pd.read_csv(metadata_csv_path)
    if isinstance(metadata["labels"].iloc[0], str):
        metadata["labels"] = metadata["labels"].apply(literal_eval)

    # Train-only, per the thesis plan's split-integrity requirement --
    # the diffusion pipeline (and now this classifier) must never touch
    # validation or test images.
    train_meta = metadata[metadata["split"] == "train"].reset_index(drop=True)

    dataset = VinDrCXRMultiLabelDataset(h5_path, train_meta, target_classes)
    n_val = max(1, int(len(dataset) * val_fraction))
    n_train = len(dataset) - n_val
    generator = torch.Generator().manual_seed(seed)
    train_subset, val_subset = random_split(dataset, [n_train, n_val], generator=generator)

    # pin_memory speeds host->GPU transfer; persistent_workers avoids
    # re-spawning (and re-opening the h5 file) every epoch.
    pin = torch.cuda.is_available()
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=pin,
                               persistent_workers=NUM_WORKERS > 0)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=pin,
                             persistent_workers=NUM_WORKERS > 0)
    return train_loader, val_loader


def train_classifier(model, train_loader, val_loader, device, num_epochs, lr, checkpoint_dir,
                      accumulation_steps=ACCUMULATION_STEPS, use_amp=USE_AMP):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = checkpoint_dir / "densenet121_mc_dropout_checkpoint.pt"
    best_path = checkpoint_dir / "densenet121_mc_dropout_best.pt"

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    model.to(device)

    # AMP only helps on CUDA; on CPU fp16 autocast is slower, so disable it.
    amp_enabled = bool(use_amp and device.type == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
    print(f"AMP: {'enabled (fp16)' if amp_enabled else 'disabled (fp32)'} | "
          f"batch {train_loader.batch_size} x {accumulation_steps} accum "
          f"= effective batch {train_loader.batch_size * accumulation_steps}")

    start_epoch = 0
    best_val_loss = float("inf")
    if ckpt_path.exists():
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        # Restore the AMP scaler too -- its loss scale is adaptive state, and
        # resuming without it can cause inf/NaN gradients in the first steps
        # after a session restart.
        if amp_enabled and ckpt.get("scaler_state") is not None:
            scaler.load_state_dict(ckpt["scaler_state"])
        start_epoch = ckpt["epoch"] + 1
        best_val_loss = ckpt.get("best_val_loss", float("inf"))
        print(f"Resumed from checkpoint: starting at epoch {start_epoch}/{num_epochs}")

    for epoch in range(start_epoch, num_epochs):
        model.train()
        train_loss, n_seen = 0.0, 0
        epoch_start = time.time()
        optimizer.zero_grad(set_to_none=True)

        for step, (images, targets, _) in enumerate(
                tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs} [train]")):
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            with torch.amp.autocast("cuda", dtype=torch.float16, enabled=amp_enabled):
                logits = model(images)
                # Divide by accumulation_steps so accumulated gradients average
                # (rather than sum) across micro-batches -- this is what makes
                # the effective batch equivalent to one large batch.
                loss = criterion(logits, targets) / accumulation_steps

            scaler.scale(loss).backward()

            is_last_batch = (step + 1) == len(train_loader)
            if (step + 1) % accumulation_steps == 0 or is_last_batch:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            # Undo the division so the logged loss stays comparable across
            # different accumulation_steps settings.
            train_loss += loss.item() * accumulation_steps * images.size(0)
            n_seen += images.size(0)
        train_loss /= max(n_seen, 1)

        model.eval()
        val_loss, n_val = 0.0, 0
        with torch.no_grad():
            for images, targets, _ in tqdm(val_loader, desc=f"Epoch {epoch + 1}/{num_epochs} [val]"):
                images = images.to(device, non_blocking=True)
                targets = targets.to(device, non_blocking=True)
                with torch.amp.autocast("cuda", dtype=torch.float16, enabled=amp_enabled):
                    loss = criterion(model(images), targets)
                val_loss += loss.item() * images.size(0)
                n_val += images.size(0)
        val_loss /= max(n_val, 1)

        elapsed = time.time() - epoch_start
        peak_gb = (torch.cuda.max_memory_allocated() / 1e9) if device.type == "cuda" else float("nan")
        print(f"Epoch {epoch + 1}/{num_epochs}: train_loss={train_loss:.4f}  "
              f"val_loss={val_loss:.4f}  {elapsed / 60:.1f} min  peak_vram={peak_gb:.1f} GB")

        is_best = val_loss < best_val_loss
        best_val_loss = min(val_loss, best_val_loss)
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scaler_state": scaler.state_dict() if amp_enabled else None,
            "best_val_loss": best_val_loss,
            "train_loss": train_loss,
            "val_loss": val_loss,
        }, ckpt_path)
        if is_best:
            torch.save(model.state_dict(), best_path)

    print(f"Training complete. Checkpoints in {checkpoint_dir}")
    return model


def train_classifier_entrypoint():
    """Run this cell to train (or resume training) the Stage 4 classifier."""
    torch.manual_seed(RANDOM_SEED)
    train_loader, val_loader = build_train_val_loaders(
        H5_PATH, METADATA_CSV_PATH, TARGET_CLASSES, BATCH_SIZE, VAL_FRACTION, RANDOM_SEED
    )
    model = build_densenet121_mc_dropout(len(TARGET_CLASSES), DROP_RATE, pretrained=True)
    model = train_classifier(model, train_loader, val_loader, DEVICE, NUM_EPOCHS, LEARNING_RATE, CHECKPOINT_DIR)
    return model


# ============================================================================
# CELL 5 — Grad-CAM (hooked on model.features, DenseNet's last conv block
# before global pooling — see torchvision.models.densenet.DenseNet.forward)
# ============================================================================
class GradCAM:
    """
    NOTE: uses activation.retain_grad() rather than
    register_full_backward_hook(). DenseNet's forward pass applies
    F.relu(features, inplace=True) immediately after model.features'
    output — a backward hook on that exact layer conflicts with the
    in-place op downstream (PyTorch raises a RuntimeError about a
    BackwardHookFunction view being modified in-place). retain_grad()
    sidesteps this entirely by reading the gradient directly off the
    activation tensor after model.backward(), with no hook-created
    autograd node in the graph. Verified empirically before finalizing.
    """
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self._fwd_handle = target_layer.register_forward_hook(self._save_activations)

    def _save_activations(self, module, inp, out):
        self.activations = out
        # `out` is a non-leaf tensor: its .grad is dropped after backward()
        # unless retained. Guarded because the "logit" path runs the same
        # model under torch.no_grad(), where requires_grad is False and
        # retain_grad() would raise.
        if out.requires_grad:
            self.activations.retain_grad()

    def remove_hooks(self):
        self._fwd_handle.remove()

    def generate_batch(self, input_batch, target_class_idx, normalize_per_pass):
        """
        BATCHED Grad-CAM: input_batch is (B, 3, H, W) -- B independent
        stochastic replicas of the SAME image (dropout draws an independent
        mask per batch element). Returns (B, h, w) float32 numpy CAMs.

        Summing the B per-sample target logits and calling backward ONCE is
        exact here, not an approximation: with BatchNorm frozen in eval mode
        samples never interact, so d(sum_i logit_i)/d(act_j) = d(logit_j)/d(act_j).
        Verified empirically to match B sequential backwards bitwise.
        """
        self.model.zero_grad(set_to_none=True)
        # autocast the forward for speed/memory, but compute the CAM itself in
        # fp32: Grad-CAM gradients are small and can underflow in fp16, which
        # would silently zero out parts of the map. No GradScaler here because
        # we never call optimizer.step() -- gradients are only read, not applied.
        with torch.amp.autocast("cuda", dtype=torch.float16,
                                 enabled=USE_AMP and input_batch.device.type == "cuda"):
            logits = self.model(input_batch)
        logits.float()[:, target_class_idx].sum().backward()

        gradients = self.activations.grad.float()      # (B, C, h, w)
        activations = self.activations.float()
        weights = gradients.mean(dim=(2, 3), keepdim=True)   # GAP over spatial dims
        cams = torch.relu((weights * activations).sum(dim=1))  # (B, h, w)
        cams = cams.detach().cpu().numpy().astype(np.float32)

        if normalize_per_pass:
            flat = cams.reshape(cams.shape[0], -1)
            cam_min = flat.min(axis=1)[:, None, None]
            cam_max = flat.max(axis=1)[:, None, None]
            span = cam_max - cam_min
            cams = np.where(span > 1e-8, (cams - cam_min) / np.maximum(span, 1e-8), 0.0)
            cams = cams.astype(np.float32)
        return cams


def get_module_by_name(model, dotted_name):
    """Resolve e.g. 'features.denseblock3' to the actual submodule."""
    modules = dict(model.named_modules())
    if dotted_name not in modules:
        raise ValueError(
            f"Layer '{dotted_name}' not found. Candidates include: "
            f"{[n for n in modules if 'denseblock' in n or n == 'features']}"
        )
    return modules[dotted_name]


@torch.no_grad()
def generate_logit_map_batch(model, input_batch, target_class_idx):
    """
    Per-pixel logit maps -- the alternative to Grad-CAM.

    DenseNet's forward is: features -> relu -> global_avg_pool -> flatten ->
    classifier. Global average pooling is what collapses the spatial map to
    a single vector. Here we SKIP the pooling and apply the classifier's
    Linear weights convolutionally (as a 1x1 conv) to the feature map, so
    every spatial location produces its own logit. Because GAP is linear,
    the spatial mean of this map equals the model's actual scalar logit --
    so this is a genuine decomposition of the prediction, not a proxy.

    Needs NO backward pass, making it roughly 3.5-4x cheaper than Grad-CAM.
    Returns (B, h, w) float32.
    """
    with torch.amp.autocast("cuda", dtype=torch.float16,
                             enabled=USE_AMP and input_batch.device.type == "cuda"):
        features = model.features(input_batch)
    features = F.relu(features.float(), inplace=False)   # fp32 from here; not inplace

    # model.classifier is nn.Sequential(nn.Dropout, nn.Linear) as built by
    # build_densenet121_mc_dropout. Pull out the Linear layer's parameters.
    linear = model.classifier[-1] if isinstance(model.classifier, nn.Sequential) else model.classifier
    weight = linear.weight[target_class_idx].view(1, -1, 1, 1)   # (1, C, 1, 1)
    bias = linear.bias[target_class_idx]

    # Apply dropout to the feature map too, so MC-Dropout stochasticity is
    # present in the classifier head, mirroring the non-convolutional path.
    if isinstance(model.classifier, nn.Sequential) and isinstance(model.classifier[0], nn.Dropout):
        features = F.dropout(features, p=model.classifier[0].p, training=True)

    logit_map = (features * weight).sum(dim=1) + bias     # (B, h, w)
    return logit_map.detach().cpu().numpy().astype(np.float32)


# ============================================================================
# CELL 6 — MC-Dropout + Grad-CAM variance map generation
# ============================================================================
def generate_mc_dropout_variance_map(model, gradcam, input_tensor, target_class_idx,
                                      n_passes=N_MC_PASSES, img_size=IMG_SIZE,
                                      method=UNCERTAINTY_METHOD,
                                      normalize_per_pass=NORMALIZE_CAM_PER_PASS,
                                      mc_batch_size=MC_BATCH_SIZE):
    """
    Runs n_passes stochastic (dropout-active, BatchNorm-frozen) passes on
    ONE image and returns (mean_map, variance_map), both (img_size,
    img_size) float32 -- matching ddim_epistemic_uncertainty_pipeline.py's
    output convention.

    Passes are executed in BATCHES of mc_batch_size replicas of the same
    image rather than one at a time (see MC_BATCH_SIZE in CONFIG for why
    this is exact). Welford's algorithm accumulates mean/variance across
    batches in O(1) memory w.r.t. n_passes.

    `method` selects what quantity is being varied:
      "gradcam" -> variance of attribution maps  (needs backward)
      "logit"   -> variance of per-pixel logits  (no backward, ~3.5-4x cheaper)
    """
    count = 0
    mean = np.zeros((img_size, img_size), dtype=np.float64)
    m2 = np.zeros((img_size, img_size), dtype=np.float64)

    passes_remaining = n_passes
    while passes_remaining > 0:
        this_batch = min(mc_batch_size, passes_remaining)
        passes_remaining -= this_batch

        # Re-assert stochastic dropout every batch, in case anything
        # upstream called .eval() on the model.
        enable_mc_dropout(model)
        input_batch = input_tensor.repeat(this_batch, 1, 1, 1)

        if method == "gradcam":
            maps = gradcam.generate_batch(input_batch, target_class_idx, normalize_per_pass)
        elif method == "logit":
            maps = generate_logit_map_batch(model, input_batch, target_class_idx)
        else:
            raise ValueError(f"Unknown UNCERTAINTY_METHOD: {method!r} (expected 'gradcam' or 'logit')")

        # Upsample the whole batch of low-res maps to img_size in one call.
        maps_resized = F.interpolate(
            torch.from_numpy(maps).unsqueeze(1),          # (B, 1, h, w)
            size=(img_size, img_size), mode="bilinear", align_corners=False,
        ).squeeze(1).numpy().astype(np.float64)           # (B, img_size, img_size)

        for single_map in maps_resized:
            count += 1
            delta = single_map - mean
            mean += delta / count
            m2 += delta * (single_map - mean)

    variance = m2 / (count - 1) if count > 1 else np.zeros_like(m2)
    return mean.astype(np.float32), variance.astype(np.float32)


def get_stage1_image_ids(cohort_source):
    """
    Recover the exact image_id set a class's Stage 1 diffusion run used, so
    Stage 4 evaluates MC-Dropout on IDENTICAL images -- required for a fair
    head-to-head comparison.

    `cohort_source` may be (see STAGE1_NPZ_DIR_BY_CLASS in CONFIG):
      - a .csv path with an `image_id` column
      - a directory of uncertainty_chunk_*.npz
      - a list/tuple of such directories (multi-session generation)

    Duplicate image_ids are de-duplicated (a set is returned sorted), but
    the count before/after is reported so a resumed session that
    reprocessed images is visible rather than silent.
    """
    # --- CSV form ---
    if isinstance(cohort_source, (str, Path)) and str(cohort_source).lower().endswith(".csv"):
        csv_path = Path(cohort_source)
        if not csv_path.exists():
            raise FileNotFoundError(f"Cohort CSV not found: {csv_path}")
        df = pd.read_csv(csv_path)
        if "image_id" not in df.columns:
            raise ValueError(
                f"{csv_path} has no 'image_id' column. Columns found: "
                f"{df.columns.tolist()}"
            )
        n_rows = len(df)
        image_ids = sorted(df["image_id"].dropna().astype(str).unique())
        if n_rows != len(image_ids):
            warnings.warn(
                f"{csv_path.name}: {n_rows} rows but only {len(image_ids)} "
                f"unique image_ids -- {n_rows - len(image_ids)} duplicate(s) "
                f"were collapsed. This is expected if sessions were resumed "
                f"and reprocessed images, but confirm it is intentional: "
                f"duplicates would double-count in any downstream statistic."
            )
        if not image_ids:
            raise RuntimeError(f"No image_ids found in {csv_path}")
        return image_ids

    # --- Directory / list-of-directories form ---
    dirs = [cohort_source] if isinstance(cohort_source, (str, Path)) else list(cohort_source)
    image_ids, total_seen = set(), 0
    for d in dirs:
        d = Path(d)
        if not d.exists():
            raise FileNotFoundError(f"Stage 1 npz directory not found: {d}")
        found_here = 0
        for p in sorted(d.glob("*.npz")):
            with np.load(p) as data:
                for key in data.files:
                    if key.endswith("__variance"):
                        image_ids.add(key[: -len("__variance")])
                        found_here += 1
        if found_here == 0:
            raise RuntimeError(f"No '*__variance' keys found under {d}")
        total_seen += found_here
        if len(dirs) > 1:
            print(f"      {found_here:5d} variance maps from {d}")

    if total_seen != len(image_ids):
        warnings.warn(
            f"{total_seen} variance maps found but only {len(image_ids)} "
            f"unique image_ids -- {total_seen - len(image_ids)} duplicate(s) "
            f"across directories. Confirm which session's copy is authoritative."
        )
    if not image_ids:
        raise RuntimeError(f"No '*__variance' keys found under {dirs}")
    return sorted(image_ids)


def generate_uncertainty_maps_for_class(model, dataset, class_name, image_ids, target_class_idx,
                                         output_dir, n_passes=N_MC_PASSES, chunk_size=CHUNK_SIZE,
                                         device=DEVICE, img_size=IMG_SIZE):
    """
    Generates MC-Dropout+Grad-CAM variance maps for every image_id, saving
    chunked uncertainty_chunk_NNN.npz + companion _meta.csv in the SAME
    format as ddim_epistemic_uncertainty_pipeline.py's Stage 1 output.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    model.to(device)

    id_to_index = {row["image_id"]: i for i, row in dataset.metadata_df.iterrows()}
    if not id_to_index:
        raise RuntimeError(
            f"The dataset passed for '{class_name}' has ZERO rows, so every "
            f"image would be skipped and no maps written. This means the "
            f"cohort image_ids did not match any row in METADATA_CSV_PATH. "
            f"Check that the metadata CSV covers the same images as the "
            f"Stage 1 cohort (and that image_id formatting matches, e.g. no "
            f"'.dicom' suffix on one side)."
        )

    # Only attach the Grad-CAM hook when it will actually be used -- the
    # "logit" path never needs it, and leaving a hook attached adds
    # pointless overhead.
    gradcam = (GradCAM(model, get_module_by_name(model, GRADCAM_TARGET_LAYER))
               if UNCERTAINTY_METHOD == "gradcam" else None)

    def _flush(chunk_dict, meta_rows, chunk_idx):
        """Write one chunk + its meta CSV. Returns the next chunk index."""
        if not meta_rows:
            return chunk_idx
        np.savez_compressed(output_dir / f"uncertainty_chunk_{chunk_idx:03d}.npz", **chunk_dict)
        pd.DataFrame(meta_rows).to_csv(
            output_dir / f"uncertainty_chunk_{chunk_idx:03d}_meta.csv", index=False
        )
        return chunk_idx + 1

    chunk_dict, meta_rows, chunk_idx = {}, [], 0
    n_processed, skipped = 0, []
    try:
        for image_id in tqdm(image_ids, desc=f"MC-Dropout maps: {class_name}"):
            if image_id not in id_to_index:
                skipped.append(image_id)
                continue

            image_tensor = dataset._load_and_resize(image_id).unsqueeze(0).to(device)
            mean_map, var_map = generate_mc_dropout_variance_map(
                model, gradcam, image_tensor, target_class_idx, n_passes, img_size
            )

            chunk_dict[f"{image_id}__mean"] = mean_map
            chunk_dict[f"{image_id}__variance"] = var_map
            meta_rows.append({
                "image_id": image_id, "finding": class_name,
                "variance_mean": float(var_map.mean()), "variance_max": float(var_map.max()),
            })
            n_processed += 1

            if len(meta_rows) >= chunk_size:
                chunk_idx = _flush(chunk_dict, meta_rows, chunk_idx)
                chunk_dict, meta_rows = {}, []
    finally:
        if gradcam is not None:
            gradcam.remove_hooks()

    # Flush any partial final chunk OUTSIDE the loop. Doing this inside the
    # loop (gated on "is this the last index?") silently loses the trailing
    # partial chunk whenever the final image happens to be skipped.
    chunk_idx = _flush(chunk_dict, meta_rows, chunk_idx)

    if skipped:
        # A printed summary, not warnings.warn: Python de-duplicates repeated
        # warnings and Kaggle often suppresses UserWarning entirely, which is
        # exactly how a total failure here can look like a silent no-op.
        print(f"  SKIPPED {len(skipped)} of {len(image_ids)} image(s) — not present in "
              f"the dataset metadata. Examples: {skipped[:5]}")

    if n_processed == 0:
        raise RuntimeError(
            f"No maps were generated for '{class_name}': all {len(image_ids)} "
            f"cohort image_ids were missing from the dataset metadata, so "
            f"nothing was written. Compare a few cohort ids against "
            f"METADATA_CSV_PATH's image_id column — this is normally an "
            f"id-format mismatch or the wrong metadata file."
        )

    print(f"Saved {chunk_idx} chunk(s), {n_processed} image(s) to {output_dir}")


def resolve_checkpoint_path(filename="densenet121_mc_dropout_best.pt"):
    """
    Decide where to load the trained model from.

    Uses INFERENCE_CHECKPOINT_PATH when set (a model trained in an earlier
    session and re-attached as a Kaggle Dataset), otherwise falls back to
    CHECKPOINT_DIR (trained in this session). INFERENCE_CHECKPOINT_PATH may
    be the .pt file itself or the directory holding it; if a directory has
    the file nested one level down -- Kaggle sometimes preserves the upload
    folder structure -- that is found too.
    """
    if INFERENCE_CHECKPOINT_PATH is not None:
        p = Path(INFERENCE_CHECKPOINT_PATH)
        if p.is_file():
            return p
        if p.is_dir():
            direct = p / filename
            if direct.exists():
                return direct
            nested = sorted(p.rglob(filename))
            if nested:
                return nested[0]
            any_pt = sorted(p.rglob("*.pt"))
            raise FileNotFoundError(
                f"INFERENCE_CHECKPOINT_PATH '{p}' contains no '{filename}'.\n"
                f"  .pt files found there: {[str(x.relative_to(p)) for x in any_pt] or 'none'}\n"
                f"  Point INFERENCE_CHECKPOINT_PATH at the .pt file directly if it "
                f"was uploaded under a different name."
            )
        raise FileNotFoundError(
            f"INFERENCE_CHECKPOINT_PATH '{p}' does not exist. Check the dataset "
            f"is attached to this notebook and the path matches the Data panel."
        )

    fallback = Path(CHECKPOINT_DIR) / filename
    if not fallback.exists():
        raise FileNotFoundError(
            f"No trained checkpoint at {fallback}.\n"
            f"  Either run train_classifier_entrypoint() first, or -- if the model "
            f"was trained in an earlier session -- set INFERENCE_CHECKPOINT_PATH to "
            f"the Kaggle Dataset holding '{filename}'. /kaggle/working does not "
            f"persist across sessions."
        )
    return fallback


def generate_uncertainty_maps_entrypoint(class_name):
    """
    Run this cell (after training completes) once per class. Loads the
    best checkpoint, reads back that class's Stage 1 image_ids, and
    generates+saves MC-Dropout variance maps to
    OUTPUT_ROOT/<class_name>/ — point calibration_spatial_evaluation_pipeline.py's
    NPZ_DIR at that folder to score it exactly like a Stage 1 run.
    """
    if class_name not in TARGET_CLASSES:
        raise ValueError(f"'{class_name}' not in TARGET_CLASSES: {TARGET_CLASSES}")
    if class_name not in STAGE1_NPZ_DIR_BY_CLASS:
        raise ValueError(
            f"No STAGE1_NPZ_DIR_BY_CLASS entry for '{class_name}' — add its "
            f"Stage 1 npz directory to CONFIG so the matched image_id set can be read."
        )

    model = build_densenet121_mc_dropout(len(TARGET_CLASSES), DROP_RATE, pretrained=False)
    best_path = resolve_checkpoint_path()
    state = torch.load(best_path, map_location=DEVICE)
    # A checkpoint trained with a different number of classes would fail with
    # torch's opaque "size mismatch for classifier.1.weight". Say what it
    # actually means: TARGET_CLASSES changed between training and now.
    ckpt_classes = state["classifier.1.weight"].shape[0]
    if ckpt_classes != len(TARGET_CLASSES):
        raise ValueError(
            f"Checkpoint '{best_path}' has {ckpt_classes} output class(es) but "
            f"TARGET_CLASSES lists {len(TARGET_CLASSES)}: {TARGET_CLASSES}. These "
            f"must match the training run exactly, in the SAME ORDER — the order "
            f"determines which output head each class maps to, and Grad-CAM/logit "
            f"maps would otherwise attribute the wrong class."
        )
    model.load_state_dict(state)
    print(f"Loaded checkpoint: {best_path}")

    image_ids = get_stage1_image_ids(STAGE1_NPZ_DIR_BY_CLASS[class_name])
    print(f"{len(image_ids)} image_ids read from Stage 1's {class_name} output.")

    metadata = pd.read_csv(METADATA_CSV_PATH)
    if isinstance(metadata["labels"].iloc[0], str):
        metadata["labels"] = metadata["labels"].apply(literal_eval)
    matched_meta = metadata[metadata["image_id"].isin(image_ids)].reset_index(drop=True)
    missing = set(image_ids) - set(matched_meta["image_id"])
    if missing:
        print(f"  {len(missing)} of {len(image_ids)} cohort image_id(s) have no row in "
              f"{METADATA_CSV_PATH}. Examples: {sorted(missing)[:5]}")
    if matched_meta.empty:
        sample_cohort = sorted(image_ids)[:3]
        sample_meta = metadata["image_id"].astype(str).head(3).tolist()
        raise RuntimeError(
            f"NONE of the {len(image_ids)} cohort image_ids for '{class_name}' "
            f"appear in METADATA_CSV_PATH — nothing could be generated.\n"
            f"  cohort ids look like:   {sample_cohort}\n"
            f"  metadata ids look like: {sample_meta}\n"
            f"If these formats differ (suffixes, case, zero-padding), fix the "
            f"mismatch. If they look the same, check METADATA_CSV_PATH points "
            f"at the metadata for this dataset and not a different one."
        )

    dataset = VinDrCXRMultiLabelDataset(H5_PATH, matched_meta, TARGET_CLASSES)
    target_class_idx = TARGET_CLASSES.index(class_name)
    # Tag by method so a "gradcam" run and a "logit" run for the same class
    # land in different folders instead of silently overwriting each other
    # (each is scored separately by Stage 3, then compared in the aggregator).
    class_slug = class_name.lower().replace("/", "_").replace(" ", "_")
    output_dir = Path(OUTPUT_ROOT) / f"{class_slug}_{UNCERTAINTY_METHOD}"

    generate_uncertainty_maps_for_class(
        model, dataset, class_name, image_ids, target_class_idx, output_dir
    )
    return output_dir


if __name__ == "__main__":
    # Kaggle usage: run train_classifier_entrypoint() in its own cell first
    # (resumable across sessions via CHECKPOINT_DIR), then, once trained,
    # call generate_uncertainty_maps_entrypoint(class_name) once per class
    # in a separate cell.
    pass

In [2]:
print("Completed model loading.")
for cls in ["Pneumothorax", "Atelectasis", "Consolidation", "Nodule/Mass", "Cardiomegaly"]:
    generate_uncertainty_maps_entrypoint(cls)

Completed model loading.
Loaded checkpoint: /kaggle/input/datasets/kartikichandratre/stage4-densenet121-mc-dropout-best-model/densenet121_mc_dropout_best.pt
65 image_ids read from Stage 1's Pneumothorax output.


MC-Dropout maps: Pneumothorax:   0%|          | 0/65 [00:00<?, ?it/s]

Saved 2 chunk(s), 65 image(s) to /kaggle/working/stage4_mc_dropout_maps/pneumothorax_logit
Loaded checkpoint: /kaggle/input/datasets/kartikichandratre/stage4-densenet121-mc-dropout-best-model/densenet121_mc_dropout_best.pt
138 image_ids read from Stage 1's Atelectasis output.


MC-Dropout maps: Atelectasis:   0%|          | 0/138 [00:00<?, ?it/s]

Saved 3 chunk(s), 138 image(s) to /kaggle/working/stage4_mc_dropout_maps/atelectasis_logit
Loaded checkpoint: /kaggle/input/datasets/kartikichandratre/stage4-densenet121-mc-dropout-best-model/densenet121_mc_dropout_best.pt
254 image_ids read from Stage 1's Consolidation output.


...

Saved 6 chunk(s), 254 image(s) to /kaggle/working/stage4_mc_dropout_maps/consolidation_logit
Loaded checkpoint: /kaggle/input/datasets/kartikichandratre/stage4-densenet121-mc-dropout-best-model/densenet121_mc_dropout_best.pt
578 image_ids read from Stage 1's Nodule/Mass output.


MC-Dropout maps: Nodule/Mass:   0%|          | 0/578 [00:00<?, ?it/s]

Saved 12 chunk(s), 578 image(s) to /kaggle/working/stage4_mc_dropout_maps/nodule_mass_logit
Loaded checkpoint: /kaggle/input/datasets/kartikichandratre/stage4-densenet121-mc-dropout-best-model/densenet121_mc_dropout_best.pt
1622 image_ids read from Stage 1's Cardiomegaly output.


MC-Dropout maps: Cardiomegaly:   0%|          | 0/1622 [00:00<?, ?it/s]

Saved 33 chunk(s), 1622 image(s) to /kaggle/working/stage4_mc_dropout_maps/cardiomegaly_logit
